In [1]:

import os

In [2]:

%pwd

'c:\\Users\\pm062\\Desktop\\End_to_End_TEXT_SUMMARIZER\\research'

In [3]:

os.chdir("../")

In [4]:

%pwd

'c:\\Users\\pm062\\Desktop\\End_to_End_TEXT_SUMMARIZER'

In [5]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataTransformationConfig:
    root_dir: Path
    data_path: Path
    tokenizer_name: str

In [6]:
from text_summarizer.constants import *
from text_summarizer.utils.common import read_yaml, create_directories

In [7]:

class ConfigurationManager:
    def __init__(
            self,
            config_filepath = CONFIG_FILE_PATH,
            params_filepath = PARAMS_FILE_PATH):
        
            self.config = read_yaml(config_filepath)
            self.params = read_yaml(params_filepath)

            create_directories([self.config.artifacts_root])

    def get_data_transformation_config(self) -> DataTransformationConfig:
        config = self.config.data_transformation

        create_directories([config.root_dir,])

        data_transformation_config = DataTransformationConfig(
            root_dir = config.root_dir,
            data_path =  config.data_path,
            tokenizer_name = config.tokenizer_name
          )
        return data_transformation_config

In [8]:

import os
from text_summarizer.logging import logger
from transformers import AutoTokenizer
from datasets import load_dataset, load_from_disk

c:\Users\pm062\Desktop\End_to_End_TEXT_SUMMARIZER\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[2026-07-11 18:12:32,172: INFO: config: PyTorch version 2.13.0 available.]


In [9]:
class DataTransformation:
    def __init__(self, config:DataTransformationConfig):
        self.config = config
        self.tokenizer = AutoTokenizer.from_pretrained(config.tokenizer_name)

    def conver_examp_to_features(self,
                                 examp_batch):
        input_encodings = self.tokenizer(
            examp_batch['dialogue'],
            max_length=1024,
            truncation=True
        )

        target_encodings = self.tokenizer(
            text_target=examp_batch['summary'],
            max_length=128,
            truncation=True
        )

        return {
            "input_ids": input_encodings["input_ids"],
            "attention_mask": input_encodings["attention_mask"],
            "labels": target_encodings["input_ids"]
        }
    
    def convert(self):
        dataset_samsum = load_from_disk(self.config.data_path)
        dataset_samsum_pt = dataset_samsum.map(self.conver_examp_to_features, batched = True)
        dataset_samsum_pt.save_to_disk(os.path.join(self.config.root_dir,"samsum_dataset"))

In [10]:
try:
    config = ConfigurationManager()
    transformation_config = config.get_data_transformation_config()
    data_transformation = DataTransformation(config=transformation_config)
    data_transformation.convert()
except Exception as e:
    raise e

[2026-07-11 18:12:32,556: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-07-11 18:12:32,559: INFO: common: yaml file: params.yaml loaded successfully]
[2026-07-11 18:12:32,561: INFO: common: created directory at:artifacts]
[2026-07-11 18:12:32,561: INFO: common: created directory at:artifacts/data_transformation]
[2026-07-11 18:12:33,258: INFO: _client: HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"]
[2026-07-11 18:12:33,591: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"]


[2026-07-11 18:12:33,592: WARNING: _http: Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.]
[2026-07-11 18:12:33,676: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/config.json "HTTP/1.1 200 OK"]
[2026-07-11 18:12:34,075: INFO: _client: HTTP Request: HEAD https://huggingface.co/google/pegasus-cnn_dailymail/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"]
[2026-07-11 18:12:34,208: INFO: _client: HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/google/pegasus-cnn_dailymail/40d588fdab0cc077b80d950b300bf66ad3c75b92/tokenizer_config.json "HTTP/1.1 200 OK"]
[2026-07-11 18:12:34,543: INFO: _client: HTTP Request: GET https://huggingface.co/api/models/google/pegasus-cnn_dailymail/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Foun

Saving the dataset (1/1 shards): 100%|██████████| 818/818 [00:00<?, ? examples/s]
